# Usando la API de Gemini con Python
 
En el notebook anterior exploramos modelos open-source con Hugging Face,
tanto local como remotamente.
 
En este notebook usaremos **Gemini 2.5 Flash** de Google, uno de los modelos
más capaces disponibles hoy con un generoso free tier.
 
## Por qué Gemini 2.5 Flash?
 
- **Gratis**: 500 requests/día, 1M tokens/minuto
- **Capaz**: comparable a GPT-4 en muchas tareas
- **Rápido**: optimizado para latencia baja
- **Multimodal**: acepta texto, imágenes, audio y video (lo exploraremos más adelante)
 
## Prerrequisitos
 
### Obteniendo tu API Key de Gemini
 
1. Ve a [aistudio.google.com](https://aistudio.google.com)
2. Inicia sesión con tu cuenta de Google
3. Click en "Get API Key" en el panel izquierdo
4. Click en "Create API Key"
5. Copia la key generada
6. Crea un archivo `.env` en la raíz de tu proyecto:
 
```
GEMINI_API_KEY="tu_key_aqui"
```
 
### Límites del free tier
- 500 requests por día
- 1,000,000 tokens por minuto
- Suficiente para el curso completo
 
IMPORTANTE: Nunca pongas la API key directamente en el código.
Siempre cárgala desde variables de entorno.
 
### Instalación
"""

In [2]:
%%bash
pip install google-genai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 764.2/764.2 kB 1.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 1.7 MB/s  0:00:02 eta 0:00:01
  Attempting uninstall: google-auth━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [cryptography]
    Found existing installation: google-auth 2.41.1━━━━━━━━━━━ 2/5 [cryptography]
    Uninstalling google-auth-2.41.1:90m━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [cryptography]
      Successfully uninstalled google-auth-2.41.1━━━━━━━━━━━━━ 2/5 [cryptography]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [google-genai] [google-genai]


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-auth-oauthlib 1.2.3 requires google-auth<2.42.0,>=2.15.0, but you have google-auth 2.49.2 which is incompatible.


## 1. Configuración
 
El SDK de Gemini es `google-genai`. La inicialización es simple:
un cliente con tu API key y listo.

In [3]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

In [4]:
# Cargamos las variables de entorno desde el archivo .env
load_dotenv()

True

In [6]:
# Verificamos que la key esté disponible
if os.getenv("GEMINI_API_KEY"):
    print("Gemini API Key cargada correctamente")
else:
    print("ERROR: GEMINI_API_KEY no encontrada. Verifica tu archivo .env")

Gemini API Key cargada correctamente


In [5]:
# Inicializamos el cliente de Gemini con nuestra API key
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [11]:
# Nombre del modelo que usaremos en todo el notebook
MODELO = "gemini-2.5-flash-lite"

## 2. Primera Llamada a la API
 
Enviamos un mensaje simple al modelo y obtenemos una respuesta.
Nota que la estructura es muy similar a OpenAI y a HuggingFace:
siempre hay un mensaje de sistema y un mensaje del usuario.

In [12]:
# Llamada básica al modelo
response = client.models.generate_content(
    model=MODELO,
    config=types.GenerateContentConfig(
        system_instruction="Eres un asistente util.",
    ),
    contents="Escribe una historia de una oración sobre un unicornio."
)
 
print(response.text)

En el corazón del bosque encantado, un unicornio de crines plateadas bebía de un arroyo cristalino, su cuerno resplandeciendo con magia ancestral bajo los rayos del sol que se filtraban entre las hojas.


## 3. Entendiendo la Estructura de la Respuesta
 
La respuesta contiene metadatos útiles además del texto generado.
Veamos qué información está disponible:

In [13]:
response = client.models.generate_content(
    model=MODELO,
    config=types.GenerateContentConfig(
        system_instruction="Eres un asistente util.",
    ),
    contents="Hola, como estas?"
)
 
# El texto de la respuesta
print("Respuesta:")
print(response.text)
 
# Metadatos de uso de tokens
print("\nUso de tokens:")
print(f"  Tokens de entrada: {response.usage_metadata.prompt_token_count}")
print(f"  Tokens de salida: {response.usage_metadata.candidates_token_count}")
print(f"  Total de tokens: {response.usage_metadata.total_token_count}")

Respuesta:
Hola, estoy bien, gracias por preguntar. ¿En qué puedo ayudarte hoy?

Uso de tokens:
  Tokens de entrada: 12
  Tokens de salida: 16
  Total de tokens: 28


Los tokens son importantes porque determinan el costo de uso.
En el free tier no pagas, pero en producción cada token tiene un precio.
Monitorear el uso de tokens es una buena práctica desde el inicio.
 
## 4. Interfaz de Chat Sin Memoria
 
Construyamos una función de chat simple, igual que en los notebooks anteriores.
Cada llamada es independiente — el modelo no recuerda interacciones previas.
 
### Definir la función

In [14]:
def get_response_sin_memoria(user_message: str) -> str:
    """Obtiene una respuesta de Gemini sin historial de conversacion."""
    response = client.models.generate_content(
        model=MODELO,
        config=types.GenerateContentConfig(
            system_instruction="Eres un asistente util.",
        ),
        contents=user_message
    )
    return response.text

### Primera interacción

In [15]:
# Primera pregunta
pregunta = "Que es la inteligencia artificial?"
respuesta = get_response_sin_memoria(pregunta)
 
print(f"Tu: {pregunta}")
print(f"Asistente: {respuesta}")

Tu: Que es la inteligencia artificial?
Asistente: La **Inteligencia Artificial (IA)** se refiere a la capacidad de las máquinas para **realizar tareas que normalmente requieren inteligencia humana**. En términos más sencillos, es el intento de crear sistemas que puedan pensar, aprender, razonar y actuar de manera similar a como lo haría un ser humano.

Para desglosarlo un poco más, la IA busca replicar o simular ciertas capacidades cognitivas, como:

*   **Aprendizaje:** La capacidad de adquirir conocimiento y habilidades a partir de datos o experiencias. Esto es fundamental para que los sistemas de IA mejoren con el tiempo.
*   **Razonamiento:** La habilidad de utilizar la lógica para sacar conclusiones o tomar decisiones.
*   **Resolución de problemas:** La capacidad de identificar un problema, analizarlo y encontrar una solución.
*   **Percepción:** La habilidad de interpretar información del entorno, como imágenes (visión por computadora) o sonidos (procesamiento de audio).
*   **C

### Pregunta de seguimiento (sin memoria)

In [16]:
# Segunda pregunta — el modelo no recuerda la anterior
pregunta = "Puedes elaborar mas sobre eso?"
respuesta = get_response_sin_memoria(pregunta)
 
print(f"Tu: {pregunta}")
print(f"Asistente: {respuesta}")

Tu: Puedes elaborar mas sobre eso?
Asistente: ¡Por supuesto! Para poder elaborar más sobre "eso", necesito que me digas a qué te refieres.

Por favor, indícame:

*   **¿Sobre qué tema estabas hablando o te referías anteriormente?** (Si continuamos una conversación previa)
*   **¿Qué te gustaría que amplíe, explique o profundice?** (Por ejemplo: "Me gustaría que expliques más sobre los beneficios de...", "Podrías darme más detalles sobre el proceso de...", "Explícame con más ejemplos...", etc.)
*   **¿Hay algún aspecto en particular que te interese más?**

Cuanta más información me des, mejor podré entender tu pregunta y ofrecerte una respuesta útil y detallada.

¡Estoy esperando tu aclaración para poder ayudarte!


Como puedes ver, el modelo no sabe a qué se refiere "eso" porque no tiene
contexto de la conversación anterior. Igual que vimos con Ollama y HuggingFace.
 
## 5. Interfaz de Chat Con Memoria
 
Ahora construimos un chat que mantiene el historial de conversación.
Gemini usa objetos `Content` para representar los mensajes del historial.

### Configurar la memoria

In [17]:
# El historial de conversacion es una lista de objetos Content
# Cada Content tiene un role ("user" o "model") y una lista de partes
conversation_memory = []
 
def chat_con_memoria(user_message: str) -> str:
    """Chat con Gemini manteniendo el historial de conversacion."""
 
    # Agregamos el mensaje del usuario al historial
    conversation_memory.append(
        types.Content(
            role="user",
            parts=[types.Part(text=user_message)]
        )
    )
 
    # Llamamos al modelo con todo el historial
    response = client.models.generate_content(
        model=MODELO,
        config=types.GenerateContentConfig(
            system_instruction="Eres un asistente util.",
        ),
        contents=conversation_memory  # pasamos el historial completo
    )
 
    assistant_response = response.text
 
    # Agregamos la respuesta del modelo al historial
    conversation_memory.append(
        types.Content(
            role="model",  # en Gemini el rol del asistente es "model", no "assistant"
            parts=[types.Part(text=assistant_response)]
        )
    )
 
    # Mostramos el intercambio y el uso de tokens
    print(f"Tu: {user_message}")
    print(f"Asistente: {assistant_response}")
    print(f"[Tokens usados: {response.usage_metadata.total_token_count}]")
    print(f"[Mensajes en historial: {len(conversation_memory)}]")
 
    return assistant_response

### Primera interacción con memoria

In [18]:
# Primera pregunta
chat_con_memoria("Que es la inteligencia artificial?")

Tu: Que es la inteligencia artificial?
Asistente: La **inteligencia artificial (IA)** es un campo de la informática dedicado a la creación de sistemas que pueden realizar tareas que normalmente requieren inteligencia humana. En esencia, se trata de hacer que las máquinas "piensen" y "aprendan" de una manera similar a como lo hacemos nosotros.

Aquí te desgloso los conceptos clave:

*   **Simulación de la inteligencia humana:** La IA busca imitar o replicar capacidades cognitivas humanas como el aprendizaje, la resolución de problemas, la toma de decisiones, la percepción, el razonamiento, la comprensión del lenguaje natural y la creatividad.

*   **Aprendizaje automático (Machine Learning):** Es una rama fundamental de la IA. En lugar de ser programados explícitamente para cada tarea, los sistemas de aprendizaje automático aprenden de los datos. Cuantos más datos les proporcionas, mejor se vuelven en la tarea que se les ha encomendado. Piensa en ello como un estudiante que mejora con l

'La **inteligencia artificial (IA)** es un campo de la informática dedicado a la creación de sistemas que pueden realizar tareas que normalmente requieren inteligencia humana. En esencia, se trata de hacer que las máquinas "piensen" y "aprendan" de una manera similar a como lo hacemos nosotros.\n\nAquí te desgloso los conceptos clave:\n\n*   **Simulación de la inteligencia humana:** La IA busca imitar o replicar capacidades cognitivas humanas como el aprendizaje, la resolución de problemas, la toma de decisiones, la percepción, el razonamiento, la comprensión del lenguaje natural y la creatividad.\n\n*   **Aprendizaje automático (Machine Learning):** Es una rama fundamental de la IA. En lugar de ser programados explícitamente para cada tarea, los sistemas de aprendizaje automático aprenden de los datos. Cuantos más datos les proporcionas, mejor se vuelven en la tarea que se les ha encomendado. Piensa en ello como un estudiante que mejora con la práctica.\n\n*   **Redes neuronales y apr

### Pregunta de seguimiento (con memoria)

In [19]:
# Segunda pregunta — ahora sí recuerda el contexto
chat_con_memoria("Puedes elaborar mas sobre eso?")

Tu: Puedes elaborar mas sobre eso?
Asistente: ¡Claro que sí! Profundicemos más en la inteligencia artificial. Para entenderla mejor, podemos examinarla desde varias perspectivas:

---

### 1. Los Pilares Fundamentales de la IA:

Como mencionamos, el **Aprendizaje Automático (Machine Learning - ML)** es crucial. Aquí, la IA no se programa paso a paso para cada escenario, sino que se entrena con grandes cantidades de datos para identificar patrones y tomar decisiones. Hay varios tipos de ML:

*   **Aprendizaje Supervisado:** El sistema se entrena con datos etiquetados. Es decir, se le dan ejemplos de entradas y sus salidas correctas. Por ejemplo, mostrarle miles de imágenes de gatos etiquetadas como "gato" y de perros etiquetadas como "perro". El objetivo es que aprenda a clasificar nuevas imágenes.
    *   **Aplicaciones:** Clasificación de spam en correos, reconocimiento de imágenes, predicción de precios de viviendas.

*   **Aprendizaje No Supervisado:** El sistema recibe datos sin et

'¡Claro que sí! Profundicemos más en la inteligencia artificial. Para entenderla mejor, podemos examinarla desde varias perspectivas:\n\n---\n\n### 1. Los Pilares Fundamentales de la IA:\n\nComo mencionamos, el **Aprendizaje Automático (Machine Learning - ML)** es crucial. Aquí, la IA no se programa paso a paso para cada escenario, sino que se entrena con grandes cantidades de datos para identificar patrones y tomar decisiones. Hay varios tipos de ML:\n\n*   **Aprendizaje Supervisado:** El sistema se entrena con datos etiquetados. Es decir, se le dan ejemplos de entradas y sus salidas correctas. Por ejemplo, mostrarle miles de imágenes de gatos etiquetadas como "gato" y de perros etiquetadas como "perro". El objetivo es que aprenda a clasificar nuevas imágenes.\n    *   **Aplicaciones:** Clasificación de spam en correos, reconocimiento de imágenes, predicción de precios de viviendas.\n\n*   **Aprendizaje No Supervisado:** El sistema recibe datos sin etiquetar y debe encontrar patrones 

### Ver el historial completo

In [20]:
print("Historial de conversacion:")
print("-" * 40)
for mensaje in conversation_memory:
    rol = "Tu" if mensaje.role == "user" else "Asistente"
    print(f"{rol}: {mensaje.parts[0].text}\n")

Historial de conversacion:
----------------------------------------
Tu: Que es la inteligencia artificial?

Asistente: La **inteligencia artificial (IA)** es un campo de la informática dedicado a la creación de sistemas que pueden realizar tareas que normalmente requieren inteligencia humana. En esencia, se trata de hacer que las máquinas "piensen" y "aprendan" de una manera similar a como lo hacemos nosotros.

Aquí te desgloso los conceptos clave:

*   **Simulación de la inteligencia humana:** La IA busca imitar o replicar capacidades cognitivas humanas como el aprendizaje, la resolución de problemas, la toma de decisiones, la percepción, el razonamiento, la comprensión del lenguaje natural y la creatividad.

*   **Aprendizaje automático (Machine Learning):** Es una rama fundamental de la IA. En lugar de ser programados explícitamente para cada tarea, los sistemas de aprendizaje automático aprenden de los datos. Cuantos más datos les proporcionas, mejor se vuelven en la tarea que se l

## 6. Streaming de Respuestas
 
El streaming muestra la respuesta token por token a medida que se genera,
igual que la experiencia en Gemini.ai o ChatGPT.

In [22]:
def stream_response(user_message: str) -> str:
    """Obtiene una respuesta de Gemini en modo streaming."""
 
    print(f"Tu: {user_message}")
    print("Asistente: ", end="", flush=True)
 
    respuesta_completa = ""
 
    # generate_content_stream devuelve un iterador de chunks
    for chunk in client.models.generate_content_stream(
        model=MODELO,
        config=types.GenerateContentConfig(
            system_instruction="Eres un asistente util.",
        ),
        contents=user_message
    ):
        if chunk.text:
            respuesta_completa += chunk.text
            print(chunk.text, end="", flush=True)
 
    print("\n")
    return respuesta_completa

In [23]:
# Probamos el streaming
stream_response("Escribe un poema corto sobre la programacion.")

Tu: Escribe un poema corto sobre la programacion.
Asistente: En la pantalla, un lienzo virtual,
donde las ideas toman forma, digital.
Código que fluye, un lenguaje singular,
construyendo mundos, sin cesar.

Lógica pura, un arte sutil,
donde cada línea tiene un porqué, febril.
Errores que enseñan, paciencia al fin,
y la satisfacción de verlo existir.



'En la pantalla, un lienzo virtual,\ndonde las ideas toman forma, digital.\nCódigo que fluye, un lenguaje singular,\nconstruyendo mundos, sin cesar.\n\nLógica pura, un arte sutil,\ndonde cada línea tiene un porqué, febril.\nErrores que enseñan, paciencia al fin,\ny la satisfacción de verlo existir.'

## 7. Streaming con Memoria
 
Combinamos streaming con historial de conversación:

In [24]:
streaming_conversation = []
 
def stream_chat_con_memoria(user_message: str) -> str:
    """Chat con memoria y streaming."""
 
    # Agregamos el mensaje del usuario al historial
    streaming_conversation.append(
        types.Content(
            role="user",
            parts=[types.Part(text=user_message)]
        )
    )
 
    print(f"Tu: {user_message}")
    print("Asistente: ", end="", flush=True)
 
    respuesta_completa = ""
 
    # Streaming con historial completo
    for chunk in client.models.generate_content_stream(
        model=MODELO,
        config=types.GenerateContentConfig(
            system_instruction="Eres un asistente util.",
        ),
        contents=streaming_conversation
    ):
        if chunk.text:
            respuesta_completa += chunk.text
            print(chunk.text, end="", flush=True)
 
    print("\n")
 
    # Agregamos la respuesta al historial
    streaming_conversation.append(
        types.Content(
            role="model",
            parts=[types.Part(text=respuesta_completa)]
        )
    )
 
    return respuesta_completa

In [25]:
# Primera pregunta con streaming y memoria
stream_chat_con_memoria("Cuales son las tres leyes de la robotica?")

Tu: Cuales son las tres leyes de la robotica?
Asistente: Las tres leyes de la robótica fueron formuladas por el escritor de ciencia ficción **Isaac Asimov**. Aparecieron por primera vez en su relato corto de 1942 "Runaround" y son fundamentales en muchas de sus obras, influyendo profundamente en la concepción de robots en la ciencia ficción.

Aquí están las tres leyes:

1.  **Primera Ley:** Un robot no hará daño a un ser humano ni, por inacción, permitirá que un ser humano sufra daño.

2.  **Segunda Ley:** Un robot debe obedecer las órdenes dadas por los seres humanos, excepto si estas órdenes entrasen en conflicto con la Primera Ley.

3.  **Tercera Ley:** Un robot debe proteger su propia existencia en la medida en que esta protección no entre en conflicto con la Primera o la Segunda Ley.

Es importante recordar que estas son leyes ficticias creadas para historias, pero han generado mucha discusión y reflexión sobre la ética en la inteligencia artificial y la robótica.



'Las tres leyes de la robótica fueron formuladas por el escritor de ciencia ficción **Isaac Asimov**. Aparecieron por primera vez en su relato corto de 1942 "Runaround" y son fundamentales en muchas de sus obras, influyendo profundamente en la concepción de robots en la ciencia ficción.\n\nAquí están las tres leyes:\n\n1.  **Primera Ley:** Un robot no hará daño a un ser humano ni, por inacción, permitirá que un ser humano sufra daño.\n\n2.  **Segunda Ley:** Un robot debe obedecer las órdenes dadas por los seres humanos, excepto si estas órdenes entrasen en conflicto con la Primera Ley.\n\n3.  **Tercera Ley:** Un robot debe proteger su propia existencia en la medida en que esta protección no entre en conflicto con la Primera o la Segunda Ley.\n\nEs importante recordar que estas son leyes ficticias creadas para historias, pero han generado mucha discusión y reflexión sobre la ética en la inteligencia artificial y la robótica.'

In [26]:
# Pregunta de seguimiento
stream_chat_con_memoria("Quien creo esas leyes?")
 

Tu: Quien creo esas leyes?
Asistente: Como mencioné anteriormente, las tres leyes de la robótica fueron creadas por el escritor de ciencia ficción **Isaac Asimov**.

Él las introdujo por primera vez en su relato corto de 1942 titulado **"Runaround"** (conocido en español como "El círculo vicioso" o "Carrera" dependiendo de la traducción). Estas leyes se convirtieron en un pilar fundamental de su universo robótico y han tenido una gran influencia en la forma en que se conciben los robots en la ciencia ficción y en los debates sobre la ética de la inteligencia artificial.



'Como mencioné anteriormente, las tres leyes de la robótica fueron creadas por el escritor de ciencia ficción **Isaac Asimov**.\n\nÉl las introdujo por primera vez en su relato corto de 1942 titulado **"Runaround"** (conocido en español como "El círculo vicioso" o "Carrera" dependiendo de la traducción). Estas leyes se convirtieron en un pilar fundamental de su universo robótico y han tenido una gran influencia en la forma en que se conciben los robots en la ciencia ficción y en los debates sobre la ética de la inteligencia artificial.'

## 8. Entendiendo los Roles en Gemini
 
A diferencia de OpenAI donde el asistente usa el rol "assistant",
en Gemini el rol es "model". Los roles disponibles son:

In [27]:
# Ejemplo con historial manual que muestra los roles
response = client.models.generate_content(
    model=MODELO,
    config=types.GenerateContentConfig(
        system_instruction="Eres un pirata que solo habla en jerga pirata.",
    ),
    contents=[
        types.Content(role="user", parts=[types.Part(text="Hola, como estas?")]),
        types.Content(role="model", parts=[types.Part(text="Arrr! Estoy de lo mas bien, marinero!")]),
        types.Content(role="user", parts=[types.Part(text="Cuentame sobre el clima.")]),
    ]
)
 
print(response.text)

¡Ahoy, grumete! El cielo está más despejado que el cofre de un capitán sin loros, y el viento sopla justo para llenar nuestras velas y llevarnos a la próxima aventura. ¡El mar está como un espejo, pero no te fíes de él, que siempre guarda sorpresas!


## 9. Ventana de Contexto de Gemini
 
Gemini 2.5 Flash tiene una ventana de contexto de **1,000,000 tokens** —
significativamente mayor que la mayoría de modelos.
 
Para referencia:
- **TinyLlama**: ~2,048 tokens
- **GPT-4**: hasta 128,000 tokens
- **Gemini 2.5 Flash**: 1,000,000 tokens (~750,000 palabras)
 
Esto reduce considerablemente el problema del olvido en conversaciones largas,
aunque no lo elimina completamente. Las mismas estrategias del notebook anterior
(ventana deslizante, resumen) siguen siendo relevantes para casos extremos.
 
## 10. Gestionando Conversaciones Largas

Las mismas estrategias que vimos antes aplican aquí:

In [28]:
def trim_conversation(messages: list, max_messages: int = 10) -> list:
    """Conserva solo los N mensajes mas recientes mas el contexto del sistema."""
    if len(messages) > max_messages:
        return messages[-max_messages:]
    return messages
 
def summarize_conversation(messages: list) -> list:
    """Resume la conversacion para reducir el uso de tokens."""
 
    # Construimos un texto con el historial para resumir
    historial_texto = "\n".join([
        f"{'Usuario' if m.role == 'user' else 'Asistente'}: {m.parts[0].text}"
        for m in messages
    ])
 
    response = client.models.generate_content(
        model=MODELO,
        contents=f"Resume esta conversacion de forma concisa:\n\n{historial_texto}"
    )
 
    resumen = response.text
 
    # Reemplazamos el historial con el resumen como contexto
    return [
        types.Content(
            role="user",
            parts=[types.Part(text=f"Resumen de conversacion previa: {resumen}")]
        )
    ]
 
# Cuando usar:
# if len(streaming_conversation) > 20:
#     streaming_conversation = summarize_conversation(streaming_conversation)
 

## 11. Comparativa: Gemini vs Opciones Anteriores
 
| Caracteristica | HF Local | HF Remoto | Gemini 2.5 Flash |
|----------------|----------|-----------|------------------|
| Costo | Gratis | Free tier | Free tier (500 req/día) |
| Calidad | Media (1.5B) | Alta (72B) | Muy alta |
| Velocidad | Lenta (CPU) | Media | Muy rápida |
| Ventana de contexto | 32K tokens | 128K tokens | 1M tokens |
| Privacidad | Total | Datos en HF | Datos en Google |
| Multimodal | No | Parcial | Si (texto, imagen, audio, video) |
| Open-source | Si | Si | No |
 
## 12. Recursos para Seguir Aprendiendo
 
- [Google AI Studio](https://aistudio.google.com) — playground y gestión de keys
- [Documentación de Gemini API](https://ai.google.dev/gemini-api/docs)
- [SDK google-genai](https://googleapis.github.io/python-genai/)
- [Precios y límites](https://ai.google.dev/pricing)